离线构造教学语言视图。核心代码、已有输出与配置沿已接受读者源；固定申报事实/图属于历史教学输入。下列新计算不构成市场策略复现。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch09_optimization_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# 第9章 — 凸优化（Convex Optimization）

本 Notebook 使用明确构造的教学输入，不含市场观测或回测。组合持有期为一年，预期简单收益以小数表示，协方差对应同一持有期。风险厌恶系数 risk_aversion=1 的单位为收益小数的倒数；改用百分点时必须同步缩放。

共享接口位于 src/quant_course/optimization_tools.py。下面直接建立最小二乘和产能分配的线性规划，便于检查变量、目标和约束。时间边界：无真实发布、信号、交易或实现收益时间；这些输出不构成可交易预测。

In [ ]:
from pathlib import Path
import sys

import cvxpy as cp
import numpy as np

ROOT = Path.cwd().resolve()
assert (ROOT / "src" / "quant_course").exists(), "Run this notebook from the quant_course project root"
sys.path.insert(0, str(ROOT / "src"))
from quant_course.optimization_tools import OptimizationError, solve_mean_variance

print("interpreter:", sys.executable)
print("CVXPY:", cp.__version__)
print("installed solvers:", cp.installed_solvers())
assert sys.prefix != sys.base_prefix, "Use the isolated reader environment"

## 两资产均值—方差二次规划（Mean–Variance Quadratic Program）

决策变量是权重。目标在方差惩罚和预期收益之间取舍，约束为完全投资及长仓。输入为可手算核对的一年期构造数值，未从市场样本估计。

In [ ]:
mu = np.array([0.05, 0.08])
sigma = np.array([[0.04, 0.01], [0.01, 0.09]])
risk_aversion = 1.0

portfolio = solve_mean_variance(mu, sigma, risk_aversion=risk_aversion)
minimum_variance = solve_mean_variance([0.0, 0.0], sigma, risk_aversion=1.0)
print("status / solver:", portfolio.solver_status, "/", portfolio.solver_name)
print("mean-variance weights:", portfolio.weights)
print("annual expected return:", portfolio.expected_return)
print("annual variance:", portfolio.variance)
print("objective:", portfolio.objective)
print("minimum-variance weights:", minimum_variance.weights)
print("minimum annual variance:", minimum_variance.variance)
np.testing.assert_allclose(portfolio.weights, [5 / 11, 6 / 11], atol=1e-7, rtol=0)
np.testing.assert_allclose(minimum_variance.weights, [8 / 11, 3 / 11], atol=1e-7, rtol=0)

## 梯度与最优性诊断

这个内点解只有预算等式绑定，故两个目标梯度分量相等。接口返回原始可行性、驻点条件（Stationarity）和互补松弛（Complementary Slackness）残差。只有原始可行性有默认阈值检查；KKT 残差只是诊断，接口未验证对偶符号或对偶间隙，不构成完整数值最优性证书。

In [ ]:
gradient = risk_aversion * sigma @ portfolio.weights - mu
print("gradient:", gradient)
print("gradient difference:", gradient[0] - gradient[1])
print("budget residual:", portfolio.budget_residual)
print("stationarity residual:", portfolio.stationarity_residual)
print("complementarity residual:", portfolio.complementarity_residual)
print("budget dual:", portfolio.budget_dual)
np.testing.assert_allclose(gradient[0], gradient[1], atol=1e-7, rtol=0)
assert portfolio.stationarity_residual < 1e-6

In [ ]:
try:
    solve_mean_variance(mu, sigma, max_weight=0.4, solver="CLARABEL")
except OptimizationError as exc:
    print("infeasible-cap result:", exc.solver_status, str(exc))
    assert exc.solver_status == "infeasible"
else:
    raise AssertionError("a 0.4 cap on both of two assets must be infeasible")

## 最小二乘（Least Squares）

四个构造响应对截距与一个解释变量回归。NumPy 和 CVXPY 都最小化残差平方和；比较系数与目标值，用线性代数解核对优化表达式。优化结构不提供回归推断或市场有效性。

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 4.0, 5.0])
A = np.column_stack([np.ones_like(x), x])

beta_np, *_ = np.linalg.lstsq(A, y, rcond=None)
beta = cp.Variable(2, name="intercept_and_slope")
ols_problem = cp.Problem(cp.Minimize(cp.sum_squares(A @ beta - y)))
ols_problem.solve()
print("[intercept, slope]:", beta.value)
print("SSE:", ols_problem.value)
np.testing.assert_allclose(beta.value, [0.9, 1.4], atol=1e-7, rtol=0)
np.testing.assert_allclose(beta.value, beta_np, atol=1e-7, rtol=0)
np.testing.assert_allclose(ols_problem.value, 0.2, atol=1e-8, rtol=0)

## 有单位的线性规划（Linear Programming）

两项变量是每日生产批次，每批贡献分别为3与2千欧元。机器能力100小时/日，人工能力80小时/日；每批占用量见代码约束。目标值单位为千欧元/日，对偶量为增加一个每日资源小时所带来的局部边际贡献。它不能跨活跃集变化无限外推。

In [ ]:
production = cp.Variable(2, nonneg=True, name="lots_per_day")
machine_limit = 2 * production[0] + production[1] <= 100  # machine-hours/day
labor_limit = production[0] + 2 * production[1] <= 80     # labor-hours/day
lp = cp.Problem(
    cp.Maximize(np.array([3.0, 2.0]) @ production),  # kEUR/day
    [machine_limit, labor_limit],
)
lp.solve(solver="CLARABEL")
print("lots/day [A, B]:", production.value)
print("contribution (kEUR/day):", lp.value)
print("binding resource use [machine h/day, labor h/day]:",
      2 * production.value[0] + production.value[1],
      production.value[0] + 2 * production.value[1])
assert lp.status == cp.OPTIMAL
np.testing.assert_allclose(production.value, [40.0, 20.0], atol=1e-6, rtol=0)
np.testing.assert_allclose(lp.value, 160.0, atol=1e-6, rtol=0)

## 目标函数几何

横轴为资产1权重，从0到1；资产2权重为1减资产1权重。纵轴为固定一年期风险—收益目标值。标记点为构造输入的内点最优解；曲率解释模型几何，不能当作历史业绩或实证有效前沿。

In [ ]:
from IPython.display import Image, display
figure_path = ROOT / "figures" / "a2" / "ch09_objective_curve.png"
display(Image(filename=str(figure_path)))